In [2]:
import numpy as np
import pandas as pd
import ssl
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

ssl._create_default_https_context = ssl._create_unverified_context

# 1. Load the dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target # Median house value in $100,000s

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("=========mse=========\n", mse)
print("=========r2=========\n", r2)

=========mse=========
 0.5558915986952444
=========r2=========
 0.5757877060324508


In [15]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load data
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create a Pipeline
# Step 1: PolynomialFeatures (degree 2 captures squares and interactions like x1*x2)
# Step 2: StandardScaler (essential because x^2 grows much faster than x)
# Step 3: LinearRegression
model_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])

# 4. Train the model
model_pipeline.fit(X_train, y_train)

# 5. Evaluate
y_pred = model_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Number of features after Polynomial expansion: {model_pipeline.named_steps['poly'].n_output_features_}")
print(f"Minimal MSE achieved: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

Number of features after Polynomial expansion: 44
Minimal MSE achieved: 0.4643
R^2 Score: 0.6457


In [19]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# 1. Load data
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names).iloc[:,:-2]
y = housing.target

# 2. Create the Pipeline
model_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    # ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])

# 3. Calculate Cross-Validation Scores
# We use 5-fold CV. 'neg_mean_squared_error' is used because sklearn 
# conventions maximize scores (higher is better), so MSE is made negative.
cv_mse_scores = cross_val_score(model_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')

# 4. Convert back to positive MSE and calculate mean
mse_scores = -cv_mse_scores
mean_mse = mse_scores.mean()
std_mse = mse_scores.std()

# 5. Final fit just to report the feature count
model_pipeline.fit(X, y)
poly_features = model_pipeline.named_steps['poly'].n_output_features_

print(f"Number of features after Polynomial expansion: {poly_features}")
print(f"Average Cross-Validated MSE: {mean_mse:.4f}")
print(f"MSE Standard Deviation: {std_mse:.4f}")

Number of features after Polynomial expansion: 27
Average Cross-Validated MSE: 3.2544
MSE Standard Deviation: 4.9672


In [23]:
pd.Series(y).describe()

count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
dtype: float64

In [24]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [21]:
cv_mse_scores

array([ -1.11457794,  -0.56766591,  -0.65886233,  -0.74919859,
       -13.18179694])